# RQ1.1 — Temporal behavior of the token embeddings

Runs the three readings of the methodology (similarity between dates, most
variable dimension, PCA trajectory) for the six per type example polygons, in the
three modes, under the strictest cloud filter (v3). Then the cloud filter
comparison of RQ1.2 on the running example.

Every figure is exported to PDF with the same publication settings as
`croma_fid_embeddings_charlotte.ipynb`.

In [ ]:
# ── 0. working directory ────────────────────────────────────────────────
# All paths in time_series_pipeline are relative (embeddings/, data_csv/,
# data_shp/), so the notebook MUST run from the repository root. Without this
# the globs return empty and every function skips silently.
import os

REPO = "/home/e2406749/tropical_forest_disturbance"
os.chdir(REPO)
assert os.path.isdir("embeddings"), f"wrong cwd: {os.getcwd()}"
assert os.path.isdir("data_csv"), f"wrong cwd: {os.getcwd()}"
assert os.path.isdir("data_shp"), f"wrong cwd: {os.getcwd()}"
print("cwd:", os.getcwd())

In [ ]:
# ── 1. publication figure config ────────────────────────────────────────
# Same settings as croma_fid_embeddings_charlotte.ipynb, cell 50.
from pathlib import Path
import matplotlib.pyplot as plt

try:
    import scienceplots  # noqa: F401
    plt.style.use(["science", "no-latex"])
except (ImportError, OSError):
    print("scienceplots not available, falling back to the default style")

TEXT_W = 160 / 25.4          # 6.30 in, text width of the document
FIGDIR = Path("figures_rq1")  # copy down to the manuscript figures/ afterwards
FIGDIR.mkdir(exist_ok=True)

plt.rcParams.update({
    "font.size": 12, "axes.labelsize": 12, "axes.titlesize": 11,
    "xtick.labelsize": 11, "ytick.labelsize": 11, "legend.fontsize": 10,
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.5,
    "savefig.bbox": "tight",
})
print("figures ->", FIGDIR.resolve())

In [ ]:
# ── 2. capture the figures the pipeline draws ───────────────────────────
# complete_analysis and compare_versions_mvd call plt.show() internally and
# neither return nor save their figures, so plt.show is wrapped to export each
# one to PDF at the document text width before displaying it.
_orig_show = plt.show
_state = {"prefix": "fig", "n": 0}


def set_prefix(prefix):
    """Name the PDFs of everything drawn from here on."""
    _state["prefix"], _state["n"] = prefix, 0


def _fit_width(fig, width=TEXT_W):
    """Scale to the text width, keeping the aspect ratio."""
    w, h = fig.get_size_inches()
    fig.set_size_inches(width, h * width / w)


def _save_and_show(*args, **kwargs):
    for num in plt.get_fignums():
        fig = plt.figure(num)
        _state["n"] += 1
        out = FIGDIR / f"{_state['prefix']}_{_state['n']:02d}.pdf"
        _fit_width(fig)
        fig.savefig(out)
        print(f"    saved {out.name}")
    _orig_show(*args, **kwargs)


plt.show = _save_and_show
print("plt.show patched")

In [ ]:
from time_series_pipeline import complete_analysis, compare_mvd

VERSION = "v3"   # strictest cloud filter, the same set the NRT analysis is scored on

# fid, tile, tok_r, tok_c   (None, None -> a random token inside the polygon)
PER_TYPE = {
    "clearcut_bare_soil":  (122, 0, None, None),
    "clearcut_vegetation": (306, 0, None, None),
    "degradation":         (389, 0, 7, 7),
    "logging_disorderly":  (371, 0, None, None),
    "fire_scar":           (367, 0, None, None),
    "logging_geometric":   (78,  0, 5, 12),
}

## Step 1 — pick the tokens

Run this first for the polygons whose token is still `None`. It draws where the
random token fell inside the polygon and prints its coordinates, so they can be
written back into `PER_TYPE` and the figures stop depending on the seed.

This is also where a bad label shows up: check the Sentinel-2 RGB grid before
committing to a polygon.

In [ ]:
for name, (fid, tile, r, c) in PER_TYPE.items():
    if r is not None:
        continue
    print(f"########## {name} — fid {fid}")
    set_prefix(f"pick_{name}_fid{fid}")
    info = complete_analysis(fid=fid, tile=tile, version=VERSION)
    print(f"  -> token ({info['tok_r']}, {info['tok_c']})")

## Step 2 — the three readings, per disturbance type

Fill the tokens found above into `PER_TYPE`, re run the definition cell, then run
this. Per polygon it produces, for each of the three modes: the $T \times T$
cosine similarity matrix (reading 1), the most variable dimension against time
(reading 2), and the PCA trajectory as PC1 alone and as PC1 to PC3 (reading 3).
Plus the NDVI profile, the Sentinel-2 RGB strip and the VH series.

In [ ]:
for name, (fid, tile, r, c) in PER_TYPE.items():
    assert r is not None, f"{name}: token still unset, run step 1"
    print(f"########## {name} — fid {fid} — token ({r},{c})")
    set_prefix(f"rq11_{name}_fid{fid}")
    complete_analysis(fid=fid, tile=tile, version=VERSION, tok_r=r, tok_c=c)

## Step 3 — cloud filter comparison (RQ1.2)

The PCA trajectory computed three times for the same token, once per filter, so
that the only quantity differing between the panels is which dates survive. One
figure per mode, each with three columns (v1, v2, v3) and three rows (most
variable dimension, PC1, PC1 to PC3).

The methodology reports the PCA rows; the MVD row is selected on the union of
the three filters and is kept here only as a check.

In [ ]:
RUNNING_EXAMPLE = "clearcut_bare_soil"
fid, tile, r, c = PER_TYPE[RUNNING_EXAMPLE]
assert r is not None, "token still unset, run step 1"

set_prefix(f"rq12_filters_fid{fid}")
compare_mvd(fid, tok_r=r, tok_c=c, tile=tile,
            modalities=("optical", "sar", "joint"))

## Notes

- `_VERSION` is a module level global in `time_series_pipeline`, written only by
  `complete_analysis` and `joint_pca_with_thumbs`, and it appears in most plot
  titles. It starts at `"v3"`, so staying on v3 throughout is safe. Check the
  title before taking any figure to the manuscript.
- The PCA base image is fitted per tile **and per mode**, so PC1 in optical and
  PC1 in joint are different axes. Only the shape of the trajectory is
  comparable between modes, never the values. Say so in the figure caption.
- Copy the PDFs to
  `manuscript_tropical_forest_disturbance/figures/` and rename the ones that
  make it into the chapter.